# MVP Playground

This notebook is a self-contained first-pass prototype for the human / robot / AI music project.

It is intentionally MIDI-only and rule-based:

- load an input MIDI file
- extract a small set of musical features
- choose a response strategy
- generate a reactive MIDI response
- write the result to disk


In [ ]:
from pathlib import Path
import numpy as np
import pretty_midi

SEED = 7
np.random.seed(SEED)

def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root from current working directory')

ROOT = find_repo_root()
INPUT_MIDI = ROOT / 'data' / 'input_midi' / 'mvp_minimalist_input.mid'
OUTPUT_MIDI = ROOT / 'data' / 'output_midi' / 'mvp_test_output.mid'
TEMPO = 120
TIME_SIG = (4, 4)
LATENCY_BARS = 1

INPUT_MIDI, OUTPUT_MIDI

## Load Input

The MVP assumes a single MIDI track for the first pass, but it will fall back to the first track if multiple instruments are present.


In [ ]:
midi = pretty_midi.PrettyMIDI(str(INPUT_MIDI))
if not midi.instruments:
    raise ValueError('Input MIDI contains no instruments')

instrument = midi.instruments[0]
notes = sorted(instrument.notes, key=lambda n: n.start)

print(f'Instruments: {len(midi.instruments)}')
print(f'Notes: {len(notes)}')
print(f'Pitch range: {min(n.pitch for n in notes)}-{max(n.pitch for n in notes)}')
print(f'Duration: {midi.get_end_time():.2f}s')


## Feature Extraction

The first version only needs a small musical summary: density, register, phrase length, and a short motif from the tail of the phrase.


In [ ]:
def seconds_per_bar(tempo=TEMPO, time_sig=TIME_SIG):
    return 60.0 / tempo * time_sig[0]

def density_bucket(note_count):
    if note_count < 24:
        return 'low'
    if note_count < 64:
        return 'medium'
    return 'high'

def register_bucket(avg_pitch):
    if avg_pitch < 50:
        return 'low'
    if avg_pitch < 70:
        return 'mid'
    return 'high'

def extract_features(notes, tempo=TEMPO, time_sig=TIME_SIG):
    pitches = [n.pitch for n in notes]
    avg_pitch = float(np.mean(pitches))
    duration = notes[-1].end - notes[0].start if notes else 0.0
    bars = max(1, int(round(duration / seconds_per_bar(tempo, time_sig))))
    tail = sorted(notes, key=lambda n: n.start)[-4:]
    motif = [n.pitch for n in tail]
    return {
        'note_count': len(notes),
        'avg_pitch': avg_pitch,
        'density': density_bucket(len(notes)),
        'register': register_bucket(avg_pitch),
        'bars': bars,
        'motif': motif,
    }

features = extract_features(notes)
features


## Response Policy

This is a simple rule table for the MVP. It is not a learned model.


In [ ]:
def decide_action(features):
    density = features['density']
    register = features['register']

    if density == 'high':
        mode = 'contrast'
        response_density = 'low'
    elif density == 'medium' and register == 'high':
        mode = 'fragment'
        response_density = 'medium'
    elif density == 'medium':
        mode = 'sequence'
        response_density = 'medium'
    else:
        mode = 'repeat'
        response_density = 'low'

    return {
        'mode': mode,
        'response_density': response_density,
        'bars': features['bars'],
        'latency_bars': LATENCY_BARS,
    }

action = decide_action(features)
action


## Generate Response

The response starts after one bar of latency and uses a simple D minor palette for the first prototype.


In [ ]:
def build_response(notes, action, tempo=TEMPO, time_sig=TIME_SIG, start_time=0.0):
    response = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    inst = pretty_midi.Instrument(
        program=pretty_midi.instrument_name_to_program('Electric Piano 1')
    )

    motif = [n.pitch for n in sorted(notes, key=lambda n: n.start)[-4:]]
    scale = [62, 65, 69, 72]
    bar_len = seconds_per_bar(tempo, time_sig)
    note_spacing = 0.5
    note_length = 0.42
    t = start_time + action['latency_bars'] * bar_len

    if action['mode'] == 'repeat' and motif:
        pitches = motif
    elif action['mode'] == 'fragment' and len(motif) >= 2:
        pitches = motif[-2:] * 2
    elif action['mode'] == 'sequence' and motif:
        pitches = motif + [p + 2 for p in motif]
    else:
        pitches = list(np.random.choice(scale, size=max(4, len(motif) or 4), replace=True))

    if action['mode'] == 'contrast':
        pitches = [max(48, min(84, p - 12)) for p in pitches]
    else:
        pitches = [max(48, min(84, p)) for p in pitches]

    if action['response_density'] == 'low':
        pitches = pitches[:max(2, len(pitches) // 2)]

    for idx, pitch in enumerate(pitches):
        note = pretty_midi.Note(
            velocity=76 if idx == 0 else 68,
            pitch=int(pitch),
            start=t,
            end=t + note_length,
        )
        inst.notes.append(note)
        t += note_spacing

    response.instruments.append(inst)
    return response

response_midi = build_response(notes, action, start_time=midi.get_end_time())
response_midi


## Write Output

The output file can be imported into a DAW or opened in a MIDI player for evaluation.


In [ ]:
combined = pretty_midi.PrettyMIDI(initial_tempo=TEMPO)
combined.instruments = midi.instruments + response_midi.instruments
OUTPUT_MIDI.parent.mkdir(parents=True, exist_ok=True)
combined.write(str(OUTPUT_MIDI))

print(f'Wrote {OUTPUT_MIDI}')
print('Detected features:', features)
print('Chosen action:', action)
